# One-session data import template

Start each figure notebook with this configuration and loader. Data-conduit discovers the selected session, parses its trial events and aligns DLC samples to VideoData timestamps. The resulting xarray dataset uses the **movement 0.17+** pose schema. Optional processing below calls movement functions.

This demo needs a real Q_C session with ExperimentEvents, VideoData and DLC files, and this checkout's data-conduit dependencies. Edit the root, directory layout, session and contents before running. No example data is substituted when a source is missing. Positions remain in **pixels**; the video-aligned acquisition clock remains in **seconds**, shared with trial/event timestamps.

In [ ]:
from pathlib import Path
import sys

# Run from this notebook folder, movement_figures, src, or the checkout root.
candidates = [Path.cwd(), *Path.cwd().parents]
source_root = next(
    (candidate for parent in candidates for candidate in (parent, parent / "src")
     if (candidate / "movement_figures").is_dir()),
    None,
)
if source_root is None:
    raise RuntimeError("Open this notebook from the data-conduit checkout.")
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

from movement_figures.data_template.loading import (
    build_config, preview_session, load_session, prepare_pose,
)

## 1. Declare root, scope and contents

The default hierarchy is `ROOT / mouseID / day / SESSION`. If the root directly contains session folders, set `LEVEL_NAMES = ()`. `SESSION` is the exact final folder name. Set `LEVEL_SELECTORS`, for example `{"l0_selector": "mouse_name"}`, to narrow intermediate folders when needed.

The figure notebooks require `trials` and `dlc`. Keep `events` to inspect raw event records; other Q_C sources can be added. VideoData is an internal DLC timing dependency, so it need not be explicitly included. This cell constructs a configuration and performs no filesystem reads.

In [ ]:
ROOT = Path("/path/to/BonsaiOutput/Training")
SESSION = "REPLACE_WITH_SESSION_FOLDER"
LEVEL_NAMES = ("mouseID", "day")
LEVEL_SELECTORS = None
SOURCES = ("trials", "events", "dlc")

config = build_config(
    ROOT,
    session=SESSION,
    level_names=LEVEL_NAMES,
    level_selectors=LEVEL_SELECTORS,
    sources=SOURCES,
)
config

## 2. Preview selection, then load

Preview reads directory names, without invoking source readers. It must find **exactly one session**; ambiguous matches require narrower selectors. The loader repeats this check before loading and checks coverage for trials, position and confidence.

In [ ]:
preview = preview_session(config)
display(preview)

In [ ]:
session_data = load_session(config)
display(session_data.loaded.session_manifest)
display(session_data.loaded.stream_coverage)
display(session_data.trials.head())
if session_data.events is not None:
    display(session_data.events.head())

In [ ]:
raw_pose = session_data.raw_pose
display(raw_pose)
print("Keypoints:", raw_pose.keypoint.values.tolist())
print("Individuals:", raw_pose.individual.values.tolist())
print("Clock range (seconds):", float(raw_pose.time[0]), float(raw_pose.time[-1]))

## 3. Choose optional movement processing

Keep `raw_pose` for comparison. The returned `pose` is a separate copy. The example confidence threshold is a starting choice to inspect and tune for the tracking output. Set it to `None` to disable filtering.

Interpolation and smoothing are disabled here. To enable bounded interpolation, set `max_gap_frames` to the maximum number of consecutive missing samples to fill. Movement interpolates by sample order, so this parameter is **not a duration in seconds**; inspect clock gaps before applying it to irregularly sampled data. A smoothing window is an odd number of samples, at least 3, for movement's rolling median. Full windows are required, preserving missing gaps.

The processing operations are recorded by movement in `pose.position.attrs["log"]`. Raw confidence scores and absolute timestamps remain available.

In [ ]:
pose = prepare_pose(
    raw_pose,
    confidence_threshold=0.9,  # Inspect/tune for this DLC recording; None disables.
    max_gap_frames=None,       # None disables interpolation.
    smoothing_window=None,     # None disables rolling median smoothing.
)
print("Missing raw coordinates:", int(raw_pose.position.isnull().sum()))
print("Missing processed coordinates:", int(pose.position.isnull().sum()))
display(pose.position.attrs)

Reuse `session_data.trials`, `session_data.events`, and `pose` in the kinematic, trajectory and spatial notebooks. The figure notebooks choose keypoints explicitly after inspecting the available names. All analytical quantities in those notebooks are computed using movement; matplotlib and xarray arrange and display the results.

This output-free template has no real-session result until the configured paths are supplied and the notebook is executed in the data environment.